# 🚗 CockpitSentinel — Train YOLO11s Distraction Model
### Unified Phone & Smoking Detection for Driver Monitoring Systems (DMS)

This notebook trains the **latest YOLO11s** model on your driver distraction dataset (`using_phone`, `smoke`, `earphone`, `person`), then exports it to **OpenVINO** (for laptop CPU) and **ONNX/TensorRT** (for vehicle edge deployment).

**Hardware Recommendation:** 
Go to **Runtime > Change runtime type > T4 GPU** (Free on Google Colab).

In [ ]:
# Step 1: Verify GPU & Install Ultralytics and dependencies
!nvidia-smi
!pip install --upgrade ultralytics roboflow openvino onnx

## Step 2: Download the Dataset
Using your Roboflow workspace (`satyapal-gaikwad`):

In [ ]:
import os
from roboflow import Roboflow

# Your Roboflow credentials
ROBOFLOW_API_KEY = "PB8gWnTTkNlkxmSOpsXH"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("satyapal-gaikwad").project("driver-smoking-phoneuse-earphone-detection-9nlii-oscoe")
version = project.version(1)
dataset = version.download(model_format="yolov11", location="/content/dataset")

print("Dataset downloaded successfully at:", dataset.location)

## Step 3: Inspect data.yaml & Classes

In [ ]:
import yaml

with open("/content/dataset/data.yaml", "r") as f:
    data_config = yaml.safe_load(f)

print("Classes:", data_config.get("names"))
print("Number of classes:", data_config.get("nc"))

## Step 4: Train YOLO11s with Transfer Learning
We use pretrained `yolo11s.pt`. Training takes ~15–20 minutes for 60 epochs on Colab T4 GPU.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 Small weights
model = YOLO("yolo11s.pt")

# Train with optimized parameters for in-cabin small-object detection
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    augment=True,
    device=0,
    name="cockpit_yolo11s_distraction",
    project="runs"
)

print("Training finished! Best weights saved at:", f"{results.save_dir}/weights/best.pt")

## Step 5: Validate and Visualize Metrics

In [ ]:
import glob
from IPython.display import Image, display

save_dir = results.save_dir
for metric_img in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    path = f"{save_dir}/{metric_img}"
    if os.path.exists(path):
        print(f"\n=== {metric_img} ===")
        display(Image(filename=path))

## Step 6: Export Model for Deployment
1. **Intel OpenVINO:** Exported for your laptop CPU (3x speedup on Intel 13th Gen).
2. **ONNX / TensorRT:** Exported for real vehicle edge computers (NVIDIA Jetson / Hailo).

In [ ]:
best_weights = f"{results.save_dir}/weights/best.pt"
best_model = YOLO(best_weights)

# 1. Export OpenVINO (FP16)
openvino_dir = best_model.export(format="openvino", half=True)
print("Exported OpenVINO:", openvino_dir)

# 2. Export ONNX
onnx_path = best_model.export(format="onnx", dynamic=True)
print("Exported ONNX:", onnx_path)

## Step 7: Download Weights to Your Laptop
Packages `best.pt`, `best_openvino_model/`, and `best.onnx` into a zip and triggers browser download.

In [ ]:
import shutil
from google.colab import files

pack_dir = "/content/cockpit_package"
os.makedirs(pack_dir, exist_ok=True)

shutil.copy(best_weights, f"{pack_dir}/best.pt")
if os.path.exists(str(openvino_dir)):
    shutil.copytree(str(openvino_dir), f"{pack_dir}/best_openvino_model", dirs_exist_ok=True)
if os.path.exists(str(onnx_path)):
    shutil.copy(str(onnx_path), f"{pack_dir}/best.onnx")

zip_file = shutil.make_archive("/content/cockpit_sentinel_yolo11s_weights", "zip", pack_dir)
print("Created download archive:", zip_file)
files.download(zip_file)